# Structured Outputs + JSON Parsing from LLM

## Problem Statement
Parse freeform SSIS error emails into validated JSON records using Groq + Pydantic.

## My Approach
Design a primary function that pairs datasets with highly structured prompts to ensure the LLM returns consistent, machine-readable JSON.
Build a secondary validate_response using Pydantic to audit the response for structural integrity, missing fields, and correct data types.

## What I Learned
Prompt formatting is the most critical factor in achieving reliable results; after multiple iterations of refining the instructions, I was able to produce a perfect output that aligned with the schema. 
I also learned that defining a clear Pydantic model is essential for catching hallucinations before they enter the database.

## Where I Got Stuck
I faced a significant hurdle where the validate_response function was consistently throwing errors across all test datasets. 
Specifically, I struggled with data type mismatches where the LLM provided strings instead of integers for the duration_minutes field.

## What I'd Do Differently
I would provide even more detailed contextual information and explicit schema rules within the prompt to ensure the LLM fully understands the transformation requirements. 
Specifically, I would emphasize the conversion logic for complex fields like duration and exit codes to prevent the validation issues I encountered during testing.

## My Solution (Naive)
*First attempt — written before reviewing feedback*

In [ ]:
from groq import Groq
from pydantic import BaseModel, validator, ValidationError
from typing import Optional, Literal
import json
import os
from dotenv import load_dotenv


load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "llama-3.1-8b-instant"  # Fast, free, good enough for Week 1


In [ ]:
#Schema Definition
class SSISErrorRecord(BaseModel):
    package_name: str
    failed_at: str
    node: str
    error_type: str
    duration_minutes: Optional[int]
    exit_code: Optional[str]
    severity: Literal["critical", "high", "medium", "low"]

In [ ]:
#Functions
def validate_response(response) -> SSISErrorRecord :
    try:
        response_json = json.loads(response)  
    except json.JSONDecodeError as e:
        print(f"--- Validation Failed: Found 1 issue ---")
        print("Malformed JSON caught:", e)
        return None 
    
    try :
        return SSISErrorRecord(**response_json)
    except ValidationError as e:
        print(f"--- Validation Failed: Found {len(e.errors())} issues ---")
        for error in e.errors():
            # Get the field name (it's a tuple, e.g., ('package_name',))
            field = " -> ".join(str(loc) for loc in error['loc'])
            error_type = error['type']
            message = error['msg']

            # Logic to distinguish the error
            if error_type == "missing":
                print(f"❌ MISSING FIELD: The field '{field}' is required but was not found.")
            else:
                # Most type mismatches start with 'type_error' or specific parsing errors
                print(f"⚠️ WRONG TYPE: The field '{field}' exists but is invalid. Error: {message} (Type: {error_type})")
        return None
    
def parse_ssis_email(email_text: str , promptbuilder) -> SSISErrorRecord:
    prompt=promptbuilder(email_text)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a senior DBA specializing in SSIS pipeline triage."},
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"}
    )
    return validate_response(response.choices[0].message.content)


In [29]:
#Test Case :-
# --- Failure Mode 1: Malformed JSON (markdown fences not stripped) ---
malformed = '```json\n{"package_name": "Test"}\n```'
validate_response(malformed)

# --- Failure Mode 2: Wrong type ---
wrong_type = '{"package_name": "Test", "failed_at": "03:17", "node": "ETL-01", "error_type": "timeout", "duration_minutes": "not-an-int", "exit_code": null, "severity": "high"}'
validate_response(wrong_type)

# --- Failure Mode 3: Missing required field ---
missing_field = '{"package_name": "Test", "failed_at": "03:17", "node": "ETL-01", "duration_minutes": 10, "exit_code": null, "severity": "high"}'
validate_response(missing_field)


--- Validation Failed: Found 1 issue ---
Malformed JSON caught: Expecting value: line 1 column 1 (char 0)
--- Validation Failed: Found 1 issues ---
⚠️ WRONG TYPE: The field 'duration_minutes' exists but is invalid. Error: Input should be a valid integer, unable to parse string as an integer (Type: int_parsing)
--- Validation Failed: Found 1 issues ---
❌ MISSING FIELD: The field 'error_type' is required but was not found.


In [ ]:
#DataSet
EMAIL_1 = """SSIS Package 'Load_Orders_Daily' failed at 03:17 AM on node ETL-PROD-02.
Error: OLE DB timeout on connection to DW_ORDERS. Duration: 1h 23m. Exit code: -1073741819."""

EMAIL_2 = """Package 'Sync_Inventory_Hourly' encountered a fatal error on ETL-DEV-01.
Flat file source not found: /data/feeds/inventory_feed.csv. No duration recorded. Exit code: 0xC020200E."""

EMAIL_3 = """SSIS: 'Archive_Customer_Logs' completed with warnings on PROD-ETL-03.
Memory threshold exceeded (82% used). Duration: 4m 10s. Exit code: 0."""


In [21]:
def few_shot_prompt(email):
    return f"""Review and convert SSIS error in json format .
Example:-
Input : SSIS Package 'Load_Orders_Daily' failed at 03:17 AM on node ETL-PROD-02. Error: OLE DB timeout on connection to DW_ORDERS. Duration: 1h 23m. Exit code: -1073741819.
Output : {{
  "package_name": "Load_Orders_Daily",
  "failed_at": "03:17 AM",
  "node": "ETL-PROD-02",
  "error_type": "OLE DB timeout",
  "duration_minutes": 83,
  "exit_code": "-1073741819",
  "severity": "critical"
}}
NOTE : for duration_minutes: integer (convert '1h 23m' → 83, '4m 10s' → 4, null if not recorded)"

Return a JSON object with schema :- 
    package_name: str
    failed_at: str
    node: str
    error_type: str
    duration_minutes: Optional[int]
    exit_code: Optional[str]
    severity: Literal["critical", "high", "medium", "low"]

  EMail:-{email}
"""
answer=parse_ssis_email(EMAIL_1 , few_shot_prompt)
print(answer)

package_name='Load_Orders_Daily' failed_at='03:17 AM' node='ETL-PROD-02' error_type='OLE DB timeout' duration_minutes=83 exit_code='-1073741819' severity='critical'


In [22]:
for email in EMAIL_1 , EMAIL_2 , EMAIL_3 :
    answer=parse_ssis_email(email , few_shot_prompt)
    print(answer)

package_name='Load_Orders_Daily' failed_at='03:17 AM' node='ETL-PROD-02' error_type='OLE DB timeout on connection to DW_ORDERS' duration_minutes=83 exit_code='-1073741819' severity='critical'
--- Validation Failed: Found 2 issues ---
⚠️ WRONG TYPE: The field 'error_type' exists but is invalid. Error: Input should be a valid string (Type: string_type)
⚠️ WRONG TYPE: The field 'exit_code' exists but is invalid. Error: Input should be a valid string (Type: string_type)
None
--- Validation Failed: Found 2 issues ---
⚠️ WRONG TYPE: The field 'failed_at' exists but is invalid. Error: Input should be a valid string (Type: string_type)
⚠️ WRONG TYPE: The field 'severity' exists but is invalid. Error: Input should be 'critical', 'high', 'medium' or 'low' (Type: literal_error)
None


## Refactored Solution

Precision Engineering: Systematically tuned prompt constraints to eliminate hallucinations and maximize relevant output.

Strategic Formatting: Implemented a hierarchical structure to improve readability and streamline the extraction of key information.

In [31]:
# Cell 5 — Prompt Builder
def few_shot_prompt_revised(email: str) -> str:
    return f"""Convert the SSIS error email below into a structured JSON object.

Example:
Input: SSIS Package 'Load_Orders_Daily' failed at 03:17 AM on node ETL-PROD-02. Error: OLE DB timeout on connection to DW_ORDERS. Duration: 1h 23m. Exit code: -1073741819.
Output: {{"package_name": "Load_Orders_Daily", "failed_at": "03:17 AM", "node": "ETL-PROD-02", "error_type": "OLE DB timeout", "duration_minutes": 83, "exit_code": "-1073741819", "severity": "critical"}}

Schema rules:
- package_name: str — name of the SSIS package
- failed_at: str — time the failure occurred (as-is from text)
- node: str — server/node name
- error_type: str — short description of error type
- duration_minutes: int — convert to total minutes, floor seconds (e.g. '1h 23m' -> 83, '4m 10s' -> 4). null if not recorded.
- exit_code: str — exit code as string, null if not present
- severity: one of 'critical', 'high', 'medium', 'low' — based on error impact

Email: {email}
"""

In [33]:
emails = {"EMAIL_1": EMAIL_1, "EMAIL_2": EMAIL_2, "EMAIL_3": EMAIL_3}

for name, email in emails.items():
    print(f"\n{'='*50}")
    print(f"Input ({name}): {email[:80].strip()}...")
    result = parse_ssis_email(email, few_shot_prompt_revised)
    if result:
        print("Parsed successfully:")
        for field, value in result.model_dump().items():
            print(f"   {field:<20} {value}")
    print()


Input (EMAIL_1): SSIS Package 'Load_Orders_Daily' failed at 03:17 AM on node ETL-PROD-02.
Error:...
Parsed successfully:
   package_name         Load_Orders_Daily
   failed_at            03:17 AM
   node                 ETL-PROD-02
   error_type           OLE DB timeout on connection to DW_ORDERS
   duration_minutes     83
   exit_code            -1073741819
   severity             critical


Input (EMAIL_2): Package 'Sync_Inventory_Hourly' encountered a fatal error on ETL-DEV-01.
Flat fi...
Parsed successfully:
   package_name         Sync_Inventory_Hourly
   failed_at            on ETL-DEV-01.
   node                 ETL-DEV-01
   error_type           Flat file source not found
   duration_minutes     None
   exit_code            0xC020200E
   severity             critical


Input (EMAIL_3): SSIS: 'Archive_Customer_Logs' completed with warnings on PROD-ETL-03.
Memory thr...
Parsed successfully:
   package_name         Archive_Customer_Logs
   failed_at            n/a
   node        